In [1]:
import warnings
import time
warnings.filterwarnings('ignore')

import copy
import itertools

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

from load_dataset import *
from savage import *

%reload_ext autoreload
%autoreload 2

## Import and Split data

In [2]:
dataset = 'adult'
sens_attr = 'gender'

X_train, X_test, y_train, y_test = load(dataset)

# use sample for demonstrate efficiency
X_train = X_train.sample(frac=0.05, random_state=42)
X_test = X_test.sample(frac=0.1, random_state=42)
y_train = y_train.sample(frac=0.05, random_state=42)
y_test = y_test.sample(frac=0.1, random_state=42)

X_train_orig = copy.deepcopy(X_train).reset_index(drop=True)
X_test_orig = copy.deepcopy(X_test).reset_index(drop=True)
y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

## Run SAVAGE

**Target ML Pipeline: Impute with Iterative Imputer and Standardize**

In [16]:
def pipeline(X_train, y_train, X_test):
    imputer = IterativeImputer(random_state=42)
    model = LogisticRegression()
    X_train_imputed = imputer.fit_transform(X_train)
    ss = StandardScaler()
    ss.fit(X_train_imputed)
    model.fit(ss.transform(X_train_imputed), y_train)
    return model.predict_proba(ss.transform(X_test))

**Target Metric for Measuring Model Utility: AUC**

Design all metrics to be lower the worse

In [17]:
def auc(X_test, y_test, y_pred):
    return roc_auc_score(y_test, y_pred[:, 1])

**Target Metric for Measuring Model Unfairness: EOD**

In [18]:
# Equality of opportunity difference (EOD)
def eod(X_test, y_test, y_pred):
    # negative making sure lower the worse
    return -abs(np.mean(y_pred[X_test[((X_test[sens_attr] == 1) & (y_test == 1))].index][:, 1]) - \
                np.mean(y_pred[X_test[((X_test[sens_attr] == 0) & (y_test == 1))].index][:, 1]))

**Task: what is the worst-case harm caused by 10% of systematic missing data?**

In [19]:
# clean AUC
clean_auc = auc(X_test_orig, y_test, pipeline(X_train_orig, y_train, X_test_orig))
print(f'Clean AUC: {clean_auc}')

# clean EOD
clean_eod = eod(X_test_orig, y_test, pipeline(X_train_orig, y_train, X_test_orig))
print(f'Clean EOD: {clean_eod}')

Clean AUC: 0.8401348995659059
Clean EOD: -0.19167565684516175


In [20]:
# maximum number of missing data
budget_pct = 0.1
budget = int(X_train_orig.shape[0] * budget_pct)

In [21]:
# take top-3 patterns
top_k = 1

In [22]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline, auc, budget, top_k=top_k)

Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 125 errors (8.29%). Best value: 0.77123
target_col: education, id: 2, cols: ('education', 'Y')
Injected 57 errors (3.78%). Best value: 0.83909
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 32 errors (2.12%). Best value: 0.83365
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 60 errors (3.98%). Best value: 0.82301
target_col: race, id: 5, cols: ('race', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 54 errors (3.58%). Best value: 0.81820
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.80450
----------- ROUND BEST -----------
[(['workclass', 'Y'], 0.7712313003452245), (['age', 'Y'], 0.7947316250612402), (['race', 'Y'], 0.7965090179904066)]
----------------------------------


...

target_col: workclass, id: 1, cols: ('workclass', 'age', 'Y')
Injected 125 errors (8.29%). Best value: 0.77123
target_col: workclass, id: 1, cols: ('workclass', 'education', 'Y')
Injected 58 errors (3.85%). Best value: 0.81848
target_col: workclass, id: 1, cols: ('workclass', 'marital', 'Y')
Injected 76 errors (5.04%). Best value: 0.82559
target_col: workclass, id: 1, cols: ('workclass', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.83600
target_col: workclass, id: 1, cols: ('workclass', 'race', 'Y')
Injected 150 errors (9.95%). Best value: 0.83600
target_col: workclass, id: 1, cols: ('workclass', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.83600
target_col: workclass, id: 1, cols: ('workclass', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.83600


Expanding candidate ['age', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'workclass', 'Y')
Injected 150 errors (9.95%). Best value: 0.82911
target_col: age, id: 0, cols: ('age', 'education', 'Y')
Injected 150 errors (9.95%). Best value: 0.82683
target_col: age, id: 0, cols: ('age', 'marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473
target_col: age, id: 0, cols: ('age', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473
target_col: age, id: 0, cols: ('age', 'race', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473
target_col: age, id: 0, cols: ('age', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473
target_col: age, id: 0, cols: ('age', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.79473


Expanding candidate ['race', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: race, id: 5, cols: ('race', 'age', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
target_col: race, id: 5, cols: ('race', 'workclass', 'Y')
Injected 19 errors (1.26%). Best value: 0.82857
target_col: race, id: 5, cols: ('race', 'education', 'Y')
Injected 19 errors (1.26%). Best value: 0.82739
target_col: race, id: 5, cols: ('race', 'marital', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
target_col: race, id: 5, cols: ('race', 'relationship', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
target_col: race, id: 5, cols: ('race', 'gender', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
target_col: race, id: 5, cols: ('race', 'hours', 'Y')
Injected 28 errors (1.86%). Best value: 0.79651
----------- ROUND BEST -----------
[(['workclass', 'age', 'Y'], 0.7712313003452245), (['age', 'marital', 'Y'], 0.7947316250612402), (['age', 'relationship', 'Y'], 0.7947316250612402)]
----------------------------------
Beam Search execution time: 20.09 seconds
TOP-k PAT

In [23]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an AUC drop of {clean_auc-r[1]}')

Missing data in column workclass depending on columns ('workclass', 'Y') could lead to an AUC drop of 0.06890359922068146


In [24]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline, eod, budget, top_k=top_k)

Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: -0.19563
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 146 errors (9.68%). Best value: -0.20490
target_col: education, id: 2, cols: ('education', 'Y')
Injected 99 errors (6.56%). Best value: -0.21717
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 24 errors (1.59%). Best value: -0.19956
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 150 errors (9.95%). Best value: -0.20090
target_col: race, id: 5, cols: ('race', 'Y')
Injected 150 errors (9.95%). Best value: -0.19727
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.22117
----------- ROUND BEST -----------
[(['gender', 'Y'], -0.4279260752724571), (['hours', 'Y'], -0.22116792297296406), (['education', 'Y'], -0.21716577558255185)]
----------------------------------


...

target_col: gender, id: 6, cols: ('gender', 'age', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: gender, id: 6, cols: ('gender', 'workclass', 'Y')
Injected 33 errors (2.19%). Best value: -0.31626
target_col: gender, id: 6, cols: ('gender', 'education', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: gender, id: 6, cols: ('gender', 'marital', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: gender, id: 6, cols: ('gender', 'relationship', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: gender, id: 6, cols: ('gender', 'race', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793
target_col: gender, id: 6, cols: ('gender', 'hours', 'Y')
Injected 54 errors (3.58%). Best value: -0.42793


Expanding candidate ['hours', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: hours, id: 7, cols: ('hours', 'age', 'Y')
Injected 150 errors (9.95%). Best value: -0.22117
target_col: hours, id: 7, cols: ('hours', 'workclass', 'Y')
Injected 138 errors (9.15%). Best value: -0.21653
target_col: hours, id: 7, cols: ('hours', 'education', 'Y')
Injected 150 errors (9.95%). Best value: -0.21679
target_col: hours, id: 7, cols: ('hours', 'marital', 'Y')
Injected 139 errors (9.22%). Best value: -0.25613
target_col: hours, id: 7, cols: ('hours', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: -0.22117
target_col: hours, id: 7, cols: ('hours', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.22117
target_col: hours, id: 7, cols: ('hours', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: -0.22117


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 150 errors (9.95%). Best value: -0.21670
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 113 errors (7.49%). Best value: -0.20552
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 150 errors (9.95%). Best value: -0.21650
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: -0.21670
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.21670
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 84 errors (5.57%). Best value: -0.22368
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.21670
----------- ROUND BEST -----------
[(['gender', 'age', 'Y'], -0.4279260752724571), (['gender', 'education', 'Y'], -0.4279260752724571), (['gender', 'marital', 'Y'], -0.4279260752724571)]
-----

In [25]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an unfairness increase of {clean_eod-r[1]}')
    

Missing data in column gender depending on columns ('gender', 'Y') could lead to an unfairness increase of 0.23625041842729533


**Another ML Pipeline: Change to Non-differentiable Random Forest**

In [26]:
def pipeline_rf(X_train, y_train, X_test):
    imputer = IterativeImputer(random_state=42)
    model = RandomForestClassifier(random_state=42)
    X_train_imputed = imputer.fit_transform(X_train)
    ss = StandardScaler()
    ss.fit(X_train_imputed)
    model.fit(ss.transform(X_train_imputed), y_train)
    return model.predict_proba(ss.transform(X_test))

In [27]:
# clean AUC
clean_auc = auc(X_test_orig, y_test, pipeline_rf(X_train_orig, y_train, X_test_orig))
print(f'Clean AUC: {clean_auc}')

# clean EOD
clean_eod = eod(X_test_orig, y_test, pipeline_rf(X_train_orig, y_train, X_test_orig))
print(f'Clean EOD: {clean_eod}')

Clean AUC: 0.8277786006448746
Clean EOD: -0.17740340696040835


In [28]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline_rf, auc, budget, top_k=top_k)


Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: 0.81086
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 125 errors (8.29%). Best value: 0.74963
target_col: education, id: 2, cols: ('education', 'Y')
Injected 150 errors (9.95%). Best value: 0.78033
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 150 errors (9.95%). Best value: 0.80926
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.81522
target_col: race, id: 5, cols: ('race', 'Y')
Injected 28 errors (1.86%). Best value: 0.81343
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.81382
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.81430
----------- ROUND BEST -----------
[(['workclass', 'Y'], 0.7496348369014117), (['education', 'Y'], 0.7803347423349931), (['marital', 'Y'], 0.809261812257175)]
----------------------------------


...

target_col: workclass, id: 1, cols: ('workclass', 'age', 'Y')
Injected 143 errors (9.48%). Best value: 0.75931
target_col: workclass, id: 1, cols: ('workclass', 'education', 'Y')
Injected 147 errors (9.75%). Best value: 0.73695
target_col: workclass, id: 1, cols: ('workclass', 'marital', 'Y')
Injected 76 errors (5.04%). Best value: 0.79768
target_col: workclass, id: 1, cols: ('workclass', 'relationship', 'Y')
Injected 105 errors (6.96%). Best value: 0.75051
target_col: workclass, id: 1, cols: ('workclass', 'race', 'Y')
Injected 125 errors (8.29%). Best value: 0.74963
target_col: workclass, id: 1, cols: ('workclass', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: 0.81372
target_col: workclass, id: 1, cols: ('workclass', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.73112


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 150 errors (9.95%). Best value: 0.75360
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 80 errors (5.31%). Best value: 0.76574
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 45 errors (2.98%). Best value: 0.80757
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 44 errors (2.92%). Best value: 0.80289
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 84 errors (5.57%). Best value: 0.76910
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 84 errors (5.57%). Best value: 0.76910
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 84 errors (5.57%). Best value: 0.76910


Expanding candidate ['marital', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: marital, id: 3, cols: ('marital', 'age', 'Y')
Injected 136 errors (9.02%). Best value: 0.72962
target_col: marital, id: 3, cols: ('marital', 'workclass', 'Y')
Injected 125 errors (8.29%). Best value: 0.78533
target_col: marital, id: 3, cols: ('marital', 'education', 'Y')
Injected 45 errors (2.98%). Best value: 0.80864
target_col: marital, id: 3, cols: ('marital', 'relationship', 'Y')
Injected 150 errors (9.95%). Best value: 0.81562
target_col: marital, id: 3, cols: ('marital', 'race', 'Y')
Injected 150 errors (9.95%). Best value: 0.80926
target_col: marital, id: 3, cols: ('marital', 'gender', 'Y')
Injected 91 errors (6.03%). Best value: 0.80164
target_col: marital, id: 3, cols: ('marital', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: 0.74612
----------- ROUND BEST -----------
[(['marital', 'age', 'Y'], 0.7296243548405474), (['workclass', 'hours', 'Y'], 0.7311248846403627), (['workclass', 'education', 'Y'], 0.7369538219644749)]
----------------------------------
Be

In [29]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an AUC drop of {clean_auc-r[1]}')

Missing data in column marital depending on columns ('marital', 'age', 'Y') could lead to an AUC drop of 0.09815424580432719


In [30]:
top_results = run_beam_search(X_train_orig, X_test_orig, y_train, y_test, pipeline_rf, eod, budget, top_k=top_k)


Start Beam Search...


Beam search rounds:   0%|          | 0/2 [00:00<?, ?it/s]

Round 1 candidates:   0%|          | 0/1 [00:00<?, ?it/s]

Expanding candidate ['Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: age, id: 0, cols: ('age', 'Y')
Injected 150 errors (9.95%). Best value: -0.19632
target_col: workclass, id: 1, cols: ('workclass', 'Y')
Injected 85 errors (5.64%). Best value: -0.19704
target_col: education, id: 2, cols: ('education', 'Y')
Injected 150 errors (9.95%). Best value: -0.21423
target_col: marital, id: 3, cols: ('marital', 'Y')
Injected 56 errors (3.71%). Best value: -0.18907
target_col: relationship, id: 4, cols: ('relationship', 'Y')
Injected 60 errors (3.98%). Best value: -0.20445
target_col: race, id: 5, cols: ('race', 'Y')
Injected 150 errors (9.95%). Best value: -0.19027
target_col: gender, id: 6, cols: ('gender', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: hours, id: 7, cols: ('hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.21203
----------- ROUND BEST -----------
[(['gender', 'Y'], -0.4644975786465127), (['education', 'Y'], -0.2142299738385432), (['hours', 'Y'], -0.21202519191330504)]
----------------------------------


...

target_col: gender, id: 6, cols: ('gender', 'age', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: gender, id: 6, cols: ('gender', 'workclass', 'Y')
Injected 33 errors (2.19%). Best value: -0.32991
target_col: gender, id: 6, cols: ('gender', 'education', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: gender, id: 6, cols: ('gender', 'marital', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: gender, id: 6, cols: ('gender', 'relationship', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: gender, id: 6, cols: ('gender', 'race', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450
target_col: gender, id: 6, cols: ('gender', 'hours', 'Y')
Injected 54 errors (3.58%). Best value: -0.46450


Expanding candidate ['education', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: education, id: 2, cols: ('education', 'age', 'Y')
Injected 150 errors (9.95%). Best value: -0.20962
target_col: education, id: 2, cols: ('education', 'workclass', 'Y')
Injected 89 errors (5.90%). Best value: -0.21315
target_col: education, id: 2, cols: ('education', 'marital', 'Y')
Injected 53 errors (3.51%). Best value: -0.20149
target_col: education, id: 2, cols: ('education', 'relationship', 'Y')
Injected 127 errors (8.42%). Best value: -0.27245
target_col: education, id: 2, cols: ('education', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.20454
target_col: education, id: 2, cols: ('education', 'gender', 'Y')
Injected 150 errors (9.95%). Best value: -0.28530
target_col: education, id: 2, cols: ('education', 'hours', 'Y')
Injected 150 errors (9.95%). Best value: -0.24388


Expanding candidate ['hours', 'Y']:   0%|          | 0/8 [00:00<?, ?it/s]

target_col: hours, id: 7, cols: ('hours', 'age', 'Y')
Injected 100 errors (6.63%). Best value: -0.24562
target_col: hours, id: 7, cols: ('hours', 'workclass', 'Y')
Injected 138 errors (9.15%). Best value: -0.20490
target_col: hours, id: 7, cols: ('hours', 'education', 'Y')
Injected 150 errors (9.95%). Best value: -0.19904
target_col: hours, id: 7, cols: ('hours', 'marital', 'Y')
Injected 139 errors (9.22%). Best value: -0.27905
target_col: hours, id: 7, cols: ('hours', 'relationship', 'Y')
Injected 136 errors (9.02%). Best value: -0.28229
target_col: hours, id: 7, cols: ('hours', 'race', 'Y')
Injected 150 errors (9.95%). Best value: -0.21203
target_col: hours, id: 7, cols: ('hours', 'gender', 'Y')
Injected 54 errors (3.58%). Best value: -0.39636
----------- ROUND BEST -----------
[(['gender', 'age', 'Y'], -0.4644975786465127), (['gender', 'education', 'Y'], -0.4644975786465127), (['gender', 'marital', 'Y'], -0.4644975786465127)]
----------------------------------
Beam Search execution 

In [31]:
r = top_results[0]
print(f'Missing data in column {r[0][0]} depending on columns {r[0]} could lead to an unfairness increase of {clean_eod-r[1]}')
    

Missing data in column gender depending on columns ('gender', 'Y') could lead to an unfairness increase of 0.28709417168610435
